<a href="https://colab.research.google.com/github/ronggobp/Machine-Learning-Course-2026/blob/main/notebooks/week-10-nlp/10_NLP_Sentiment_Analysis_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Setup & Import Data

In [ ]:
import pandas as pd
import numpy as np
import re
import string
import tensorflow_datasets as tfds
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
import wandb

# 1. Inisialisasi W&B
run = wandb.init(project="nlp-sentiment-analysis", name="tfidf-naive-bayes-tfds")

# 2. Load Dataset IMDB dari TensorFlow Datasets (Sangat Stabil)
print("Sedang mengunduh dataset IMDB dari TensorFlow... (Mohon tunggu)")
ds, info = tfds.load('imdb_reviews', with_info=True, as_supervised=True)

# 3. Konversi ke Pandas DataFrame agar sesuai dengan alur notebook Anda
train_data = list(tfds.as_numpy(ds['train']))
df = pd.DataFrame(train_data, columns=['review', 'label'])

# Decode bytes ke string dan ubah label angka menjadi teks (0: neg, 1: pos)
df['review'] = df['review'].str.decode("utf-8")
df['label'] = df['label'].map({0: 'neg', 1: 'pos'})

# Ambil 10.000 sampel agar proses training tetap cepat
df = df.sample(10000, random_state=42)

print(f"Dataset Berhasil Dimuat! Jumlah data: {len(df)}")
df.head()

Text Cleaning

In [ ]:
def clean_text(text):
    text = text.lower() # Kecilkan semua
    text = re.sub('\[.*?\]', '', text) # Hapus teks dalam kurung
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text) # Hapus tanda baca
    text = re.sub('\w*\d\w*', '', text) # Hapus kata yang mengandung angka
    return text

df['review_clean'] = df['review'].apply(clean_text)
print("Contoh teks sebelum & sesudah dibersihkan:")
print(df[['review', 'review_clean']].iloc[0])

Vectorization

In [8]:
# TF-IDF memberikan bobot lebih pada kata yang penting dan langka
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['review_clean'])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

wandb.log({"vectorizer_type": "TF-IDF", "vocab_size": 5000})

Training & Evaluation

In [ ]:
model = MultinomialNB()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"Akurasi Sentimen: {acc:.4f}")
print(classification_report(y_test, y_pred))

wandb.log({"Accuracy": acc})
wandb.finish()